In [1]:
import pandas as pd
import networkx as nx
import numpy as np

In [2]:
df = pd.read_csv("../data/raw/delivery_data.csv")

print(df.shape)

(144867, 24)


In [3]:
edge_df = (
    df.groupby(
        ["source_center", "destination_center"]
    )
    .agg(
        median_factor=("segment_factor", "median"),
        trip_count=("trip_uuid", "count")
    )
    .reset_index()
)

edge_df.head()

,source_center,destination_center,median_factor,trip_count
0,IND000000AAL,IND411033AAA,2.214286,37
1,IND000000AAQ,IND700028AAB,4.939394,4
2,IND000000AAS,IND783370AAC,1.833333,18
3,IND000000AAZ,IND444203AAA,3.208333,3
4,IND000000AAZ,IND444303AAA,2.043478,3


In [4]:
G = nx.DiGraph()

for _, row in edge_df.iterrows():

    G.add_edge(
        row["source_center"],
        row["destination_center"],
        weight=row["median_factor"],
        volume=row["trip_count"]
    )

print(
    "Nodes:",
    G.number_of_nodes()
)

print(
    "Edges:",
    G.number_of_edges()
)

Nodes: 1657
Edges: 2783


In [5]:
betweenness = nx.betweenness_centrality(
    G,
    normalized=True
)

pagerank = nx.pagerank(
    G
)

in_degree = dict(G.in_degree())

out_degree = dict(G.out_degree())

In [6]:
node_features = pd.DataFrame({

    "node":
        list(G.nodes()),

    "betweenness":
        [betweenness.get(n,0)
         for n in G.nodes()],

    "pagerank":
        [pagerank.get(n,0)
         for n in G.nodes()],

    "in_degree":
        [in_degree.get(n,0)
         for n in G.nodes()],

    "out_degree":
        [out_degree.get(n,0)
         for n in G.nodes()]
})

node_features.head()

,node,betweenness,pagerank,in_degree,out_degree
0,IND000000AAL,0.000000,0.000464,1,1
1,IND411033AAA,0.043254,0.005901,23,20
2,IND000000AAQ,0.000000,0.000119,0,1
3,IND700028AAB,0.000404,0.000579,2,1
4,IND000000AAS,0.003225,0.000515,1,1


In [7]:
node_features.to_csv(
    "../data/processed/node_features.csv",
    index=False
)

print("saved")

saved
